# Milestone 1: ADM2 + WorldPop Under-18 Population Fusion

This notebook creates the first fused spatial dataset for ChildReach. It aggregates WorldPop 2019 under-age-18 raster values into Mozambique ADM2 boundary polygons using zonal statistics.

The output is a district-level GeoPackage and CSV with:

- estimated under-18 population (`under18_sum`)
- the number of valid WorldPop raster cells used (`valid_cell_count`)
- a transparent raster-support flag (`worldpop_valid_cells`)


## 1. Load libraries and input paths

The ADM2 boundaries define the analysis units. The WorldPop raster provides gridded under-18 population estimates. Both sources were previously inspected for CRS, bounds, NoData handling, and value plausibility.


In [ ]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

import geopandas as gpd
import pandas as pd
from rasterstats import zonal_stats

from childreach.paths import (
    ADM2_BOUNDARIES_GEOJSON,
    DATA_PROCESSED,
    MOZ_ADM2_UNDER18_2019_CSV,
    MOZ_ADM2_UNDER18_2019_GPKG,
    WORLDPOP_UNDER18_TOTAL_2019_TIF,
)

boundaries_path = ADM2_BOUNDARIES_GEOJSON
raster_path = WORLDPOP_UNDER18_TOTAL_2019_TIF
processed_dir = DATA_PROCESSED
processed_dir.mkdir(parents=True, exist_ok=True)

gpkg_path = MOZ_ADM2_UNDER18_2019_GPKG
csv_path = MOZ_ADM2_UNDER18_2019_CSV


## 2. Load ADM2 boundaries

Expected result: 159 ADM2 district/district-like polygons in EPSG:4326.


In [ ]:
boundaries = gpd.read_file(boundaries_path)

print(boundaries.shape)
print(boundaries.crs)
print(boundaries[["shapeName", "shapeID", "shapeType", "shapeGroup"]].head())


## 3. Run zonal statistics

For each ADM2 polygon, `zonal_stats` summarizes valid raster cells inside the polygon.

- `sum` becomes the estimated under-18 population total for the district.
- `count` is the number of valid raster cells contributing to that estimate.
- `nodata=-99999` prevents WorldPop missing-value cells from being treated as population values.


In [ ]:
stats = zonal_stats(
    boundaries,
    raster_path,
    stats=["sum", "count"],
    nodata=-99999,
)

stats_df = pd.DataFrame(stats)

print(stats_df.head())
print(stats_df.shape)
print(stats_df.isna().sum())


## 4. Join statistics back to ADM2 boundaries

Keep missing population estimates as `NaN` instead of converting them to zero. A missing value means there were no valid WorldPop raster cells for that polygon, not that the true under-18 population is zero.


In [ ]:
results = boundaries.copy()
results["under18_sum"] = stats_df["sum"]
results["valid_cell_count"] = stats_df["count"]
results["worldpop_valid_cells"] = results["valid_cell_count"] > 0

print(results[["shapeName", "under18_sum", "valid_cell_count", "worldpop_valid_cells"]].head())
print(results["worldpop_valid_cells"].value_counts(dropna=False))
print(results["under18_sum"].describe())


## 5. Diagnose missing raster support

Two island ADM2 features have no valid WorldPop cells. This diagnostic keeps that limitation visible for the paper, map legend, and future app.


In [ ]:
missing = results.loc[
    ~results["worldpop_valid_cells"],
    ["shapeName", "shapeID", "shapeType", "valid_cell_count", "under18_sum"],
]

print(missing)
print(missing["valid_cell_count"].describe())


## 6. Check whether `all_touched=True` changes the missing-island diagnosis

This is a focused diagnostic, not the primary analysis setting. If the islands still have zero valid cells with `all_touched=True`, the issue is raster support/coverage rather than only the default cell-center rule.


In [ ]:
stats_all_touched = zonal_stats(
    boundaries,
    raster_path,
    stats=["sum", "count"],
    nodata=-99999,
    all_touched=True,
)

stats_all_touched_df = pd.DataFrame(stats_all_touched)

comparison = boundaries[["shapeName", "shapeID"]].copy()
comparison["count_default"] = stats_df["count"]
comparison["sum_default"] = stats_df["sum"]
comparison["count_all_touched"] = stats_all_touched_df["count"]
comparison["sum_all_touched"] = stats_all_touched_df["sum"]

print(
    comparison.loc[
        comparison["shapeName"].isin(["Ilha Licom", "Ilha Risunodo"]),
        [
            "shapeName",
            "count_default",
            "sum_default",
            "count_all_touched",
            "sum_all_touched",
        ],
    ]
)


## 7. Sanity-check highest and lowest valid ADM2 totals

This is not a formal validation against census totals. It is a plausibility check: large urban/populous districts should generally appear near the high end, while sparse or small districts should appear near the low end.


In [ ]:
valid_results = results.loc[
    results["worldpop_valid_cells"],
    ["shapeName", "under18_sum", "valid_cell_count"],
]

print("Highest under-18 estimates")
print(valid_results.sort_values("under18_sum", ascending=False).head(10))

print("\nLowest under-18 estimates")
print(valid_results.sort_values("under18_sum", ascending=True).head(10))


## 8. Save processed outputs

The GeoPackage preserves geometry for future mapping and spatial analysis. The CSV provides a lightweight table for inspection, reporting, and non-spatial analysis.


In [ ]:
results.to_file(gpkg_path, layer="moz_adm2_under18_2019", driver="GPKG")
results.drop(columns="geometry").to_csv(csv_path, index=False)

print(gpkg_path)
print(gpkg_path.exists(), gpkg_path.stat().st_size)
print(csv_path)
print(csv_path.exists(), csv_path.stat().st_size)


## 9. Reload outputs to verify reproducibility

A successful reload confirms the saved files can be reused by later notebooks, scripts, maps, reports, or an app prototype.


In [ ]:
processed_gdf = gpd.read_file(gpkg_path, layer="moz_adm2_under18_2019")
processed_csv = pd.read_csv(csv_path)

print(processed_gdf.shape)
print(processed_csv.shape)
print(processed_gdf.crs)
print(processed_gdf[["shapeName", "under18_sum", "valid_cell_count", "worldpop_valid_cells"]].head())
print(processed_gdf["worldpop_valid_cells"].value_counts(dropna=False))


## Milestone 1 interpretation

This notebook completes the first ChildReach spatial fusion milestone: ADM2 boundaries plus WorldPop under-18 raster data have been combined into a district-level analysis dataset.

The resulting values should be interpreted as aggregate, model-based estimates. Two island ADM2 features are retained with missing under-18 totals because no valid WorldPop raster cells supported those polygons. This project supports further humanitarian assessment; it does not determine aid allocation or identify individual children.
